goal is to answer these questions

1What is the monthly revenue, and how much did it grow month over month?


q2Which 10 product categories earn the most?


q3Which states have the highest total revenue and highest average order value?

q4What is the average delivery time, and what % of orders arrived late, by state?

q5Do late deliveries get lower review scores?

q6What is the payment method mix, and how many installments does each method use on average?

q7What % of customers ordered more than once?

q8Which sellers earn the most, and which of those have the worst reviews?

q9On which day of the week and at what hour do orders peak?

q10What share of total revenue comes from the top 10% of customers?


In [2]:
import pandas as pd
##knowing the data
datas =["olist_customers_dataset.csv","olist_geolocation_dataset.csv","olist_order_items_dataset.csv",
        "olist_order_payments_dataset.csv","olist_order_reviews_dataset.csv","olist_orders_dataset.csv",
        "olist_products_dataset.csv","olist_sellers_dataset.csv","product_category_name_translation.csv"]

for i in datas:
    data = pd.read_csv(i)
    print(i)
    print(f"information of {i} :\n :{data.info()} \n \n ")
    print(f"no of empty cell:{data.isna().sum()}\n \n ")
    print(f"first 2 rows of {i} \n {data.head(2)} \n \n")
    print("\n ends \n \n")

olist_customers_dataset.csv
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB
information of olist_customers_dataset.csv :
 :None 
 
 
no of empty cell:customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
 
 
first 2 rows of olist_customers_dataset.csv 
                         customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b

In [3]:
###few customers ordered again
customers = pd.read_csv("olist_customers_dataset.csv")
no_of_unique_customerid = customers["customer_unique_id"].nunique()
no_of_customersid = customers["customer_id"].nunique()
print(f"no of unique customer id is = {no_of_unique_customerid}")
print(f"no of customers id is = {no_of_customersid}")

no of unique customer id is = 96096
no of customers id is = 99441


In [4]:
geo = pd.read_csv("olist_geolocation_dataset.csv")
##geoloc is heavily repetative
no_ofzipcodes= geo["geolocation_zip_code_prefix"].nunique()
print(no_ofzipcodes)

19015


In [5]:
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
uniqueorderids= reviews["order_id"].nunique()
print(uniqueorderids)

98673


 Day 1 Findings

- Orders: date columns are stored as text (str), not real dates. Need to convert.
- Customers: customer_id is per order (99,441 unique), customer_unique_id is per person (96,096 unique). 
  Used customer_unique_id to find repeat customers.
- Geolocation: only 19,015 unique zip prefixes across 1,000,163 rows. Heavily duplicated, 
  will cause row explosion if merged on zip without care.
- Reviews: 98,673 unique order_id values vs 99,441 orders. Some orders have 2+ reviews, 
  others have none.
- Products: column names product_name_lenght and product_description_lenght have a typo 
  (should be "length").

In [6]:
###converting date as str to real dates
orders = pd.read_csv("olist_orders_dataset.csv")
orders["order_purchase_timestamp"]= pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])
orders["order_delivered_customer_date"] =  pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])
print(orders.info())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB
None


2965 empty cells in deliverey 

In [7]:
# #data
typesoforderstatus = print(orders["order_status"].value_counts())
# some orders are marked as delivered but not reached to customers
orders[(orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].isna()]
##2965 orders did not have a date and here 2963 orders are eithr pending or cancelled and 8 delivered ordeers
#doesnot have delivery dat so 2963+8-2965 = 6 non delivered orders have a date
for i in ["shipped",
"canceled",
"unavailable",
"invoiced",
"processing",
"created",
"approved"]:
 print(orders[(orders["order_status"] == i) & orders["order_delivered_customer_date"].notna()])
 #6cancelled orders have a delivery date


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
Empty DataFrame
Columns: [order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date]
Index: []
                               order_id                       customer_id  \
2921   1950d777989f6a877539f53795b4c3c3  1bccb206de9f0f25adc6871a1bcf77b2   
8791   dabf2b0e35b423f94618bf965fcb7514  5cdec0bb8cbdf53ffc8fdc212cd247c6   
58266  770d331c84e5b214bd9dc70a10b829d0  6c57e6119369185e575b36712766b0ef   
59332  8beb59392e21af5eb9547ae1a9938d06  bf609b5741f71697f65ce3852c5d2623   
92636  65d1e226dfaeb8cdc42f665422522d14  70fc57eeae292675927697fe03ad3ff5   
94399  2c45c33d2f9cb8ff8b1c86cc28c11c30  de4caa97afa80c8eeac2ff4c8da5b72e   

      order_status order_purchase_timestamp

In [8]:
## approved_at has 160 missing values
missing_approved_delivered = orders[
    orders["order_approved_at"].isna() & (orders["order_status"] == "delivered")
]

print(f"Delivered orders with missing approved_at: {missing_approved_delivered.shape[0]}")
missing_approved_delivered.head()
##order_delivered_carrier_date has 1783 missing values
missing_carrier_deliveres= orders[
    orders["order_delivered_carrier_date"].isna() & (orders["order_status"] == "delivered")
]

print(f"Delivered orders with missing carrier date: {missing_carrier_deliveres.shape[0]}")
missing_carrier_deliveres.head()


Delivered orders with missing approved_at: 14
Delivered orders with missing carrier date: 2


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23


In [9]:
products = pd.read_csv("olist_products_dataset.csv")
#renaming product_name_lenght         32341 non-null  float64
 #3   product_description_lenght
products = products.rename(columns= {"product_name_lenght":"product_name_length","product_description_lenght":"product_description_length"})
print(products.head())

                         product_id  product_category_name  \
0  1e9e8ef04dbcff4541ed26657ea517e5             perfumaria   
1  3aa071139cb16b67ca9e5dea641aaa2f                  artes   
2  96bd76ec8810374ed1b65e291975717f          esporte_lazer   
3  cef67bcfe19066a932b7673e239eb23d                  bebes   
4  9dc1a7de274444849c219cff195d0b71  utilidades_domesticas   

   product_name_length  product_description_length  product_photos_qty  \
0                 40.0                       287.0                 1.0   
1                 44.0                       276.0                 1.0   
2                 46.0                       250.0                 1.0   
3                 27.0                       261.0                 1.0   
4                 37.0                       402.0                 4.0   

   product_weight_g  product_length_cm  product_height_cm  product_width_cm  
0             225.0               16.0               10.0              14.0  
1            1000.0       

1)8 delivered orders have no delivery date and 6 canceled orders have one. The delivery analysis (Q4, Q5) will use only orders with status delivered and a valid delivery date.

2)14 delivered orders have no approval date and 2 have no carrier date. No question uses those columns  i left them as they are.

3)All other gaps are expected: the status explains why the dates are empty

4)renamed lenght to length
5)converted date str in date us

In [10]:
##monthly revenue columns needed are 
#from orders dataset
#  ##purchase time date  #order id
#from order payments need payvalue


payments = pd.read_csv("olist_order_payments_dataset.csv")
# print(payments.info())
##numbers of order id =! numbers of order id in payments
order_items = pd.read_csv("olist_order_items_dataset.csv")
###all three data have diff numbers of order id 
#order id that r present in all three


In [11]:
print(payments.info())

print(payments.nunique())
#payment has unique 99440 values while orders has 99441
print(orders.info())
##orders have no duplicate order id
print(orders.nunique())
print(order_items.info())
print(order_items.nunique())
#98666 unique values in orderitems
#if same then i can simply create a new df with order id and pay value and date and maybe customer id
#have to check orderr ids that are not in payment and vice versa


<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB
None
order_id                99440
payment_sequential         29
payment_type                5
payment_installments       24
payment_value           29077
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer

In [12]:
orders[~orders['order_id'].isin(payments['order_id'])]
# 'all order ids are present in payments order id 
# ' except 1 of 99,441 orders (a delivered order) has no payment record'index no30710,30711,1
#order id 	bfbd0f9bdef84302105ad712db648a6c

##orders have only succsfull orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04


In [22]:
#there is a diff between orders in orders and order items lets check
uniqueorderids = orders[~orders['order_id'].isin(order_items['order_id'])]
print(uniqueorderids['order_status'].count())
#775 ordedr unique in orders ids
print(uniqueorderids["order_status"].value_counts())
##thes 775 orders were never reached to our mian revenue system and they really don't
#affect our questions cause these 775 don't have much informations  not price nor product

775
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64
